In [5]:
"""
# =============================================================================
# GLOBAL STABILITY ANALYSIS — SHIFT-INVERT EIGENSOLVER
#
# Pipeline:
#   1. Read sparse Jacobian from .npz (CSR) format
#   2. Solve eigenproblem via SLEPc shift-invert (Krylov-Schur)
#   3. Report and save converged eigenpairs
#
# Requires: petsc-complex environment (PETSc/SLEPc complex scalar build)
#
# MUMPS out-of-core (OOC):
#   Enabled by default for large meshes where the LU factorisation
#   exceeds available RAM. MUMPS spills factor blocks to disk and
#   swaps them in/out during the solve.
#   → Point ooc_dir to a fast disk with sufficient free space (SSD preferred)
#   → Tune icntl_23 = total_RAM_MB / np  (max RAM per MPI process)
#
# Tunable parameters:
#   sigma      : complex shift  — place near expected physical eigenvalue
#   nev        : number of eigenvalues requested
#   ncv        : Krylov subspace size  (default: max(3*nev, 30))
#   tol        : eigensolver convergence tolerance  (default 1e-10)
#   ooc_dir    : directory for MUMPS temporary OOC files
#   icntl_14   : MUMPS extra working memory headroom in %  (default 80)
#   icntl_23   : MUMPS max RAM per MPI process in MB
# =============================================================================


Requires the complex-scalar PETSc/SLEPc build (petsc-complex env),
since sigma is a general complex number here.

Adding Out of core capability to the MUMPs LU factorisation to prevent crashing

"""
import numpy as np
import matplotlib.pyplot as plt
import scipy.sparse as sp
from petsc4py import PETSc
from slepc4py import SLEPc

import os

# =====================================================================
# 1. Read the Jacobian
# =====================================================================
def read_jacobian(path):
    """Load a sparse Jacobian saved via scipy.sparse.save_npz.
    Returns a CSR matrix cast to PETSc's scalar type (complex128
    in this environment)."""
    J = sp.load_npz(path)
    J = J.tocsr().astype(PETSc.ScalarType)
    print(f"Loaded Jacobian: shape={J.shape}, nnz={J.nnz}, "
          f"density={J.nnz / (J.shape[0]*J.shape[1]):.2e}")
    return J


def scipy_csr_to_petsc(J_csr):
    Mat = PETSc.Mat().createAIJ(size=J_csr.shape,
                                  csr=(J_csr.indptr, J_csr.indices, J_csr.data))
    Mat.assemble()
    return Mat


# =====================================================================
# 2. Solve with shift-invert
# =====================================================================
def solve_shift_invert(J, sigma, nev=10, ncv=None, tol=1e-10, max_it=2000):
    n = J.getSize()[0]
    if ncv is None:
        ncv = min(n, max(3 * nev, 30))
    E = SLEPc.EPS().create()
    E.setOperators(J)
    E.setProblemType(SLEPc.EPS.ProblemType.NHEP)
    E.setType(SLEPc.EPS.Type.KRYLOVSCHUR)
    E.setDimensions(nev=nev, ncv=ncv)
    E.setTolerances(tol=tol, max_it=max_it)
    st = E.getST()
    st.setType(SLEPc.ST.Type.SINVERT)
    st.setShift(sigma)
    ksp = st.getKSP()
    ksp.setType('preonly')
    pc = ksp.getPC()
    pc.setType('lu')
    try:
        pc.setFactorSolverType('mumps')
        solver_used = 'mumps'
    except PETSc.Error:
        pc.setFactorSolverType('petsc')
        solver_used = 'petsc (built-in, no mumps found)'

    # ── MUMPS out-of-core — add here ──────────────────────────────────────────
    ooc_dir = "/mnt/data1/ahf25/"     # ← point to your large disk
    os.makedirs(ooc_dir, exist_ok=True)
    os.environ['MUMPS_OOC_TMPDIR'] = ooc_dir
    opts = PETSc.Options()
    opts['mat_mumps_icntl_22'] = 1     # enable OOC
    opts['mat_mumps_icntl_14'] = 80    # +80% working memory headroom
    opts['mat_mumps_icntl_23'] = 12000 # max RAM per process in MB
    # ─────────────────────────────────────────────────────────────────────────

    E.setTarget(sigma)
    E.setWhichEigenpairs(SLEPc.EPS.Which.TARGET_MAGNITUDE)
    E.setFromOptions()
    print(f"Solving: sigma={sigma}, nev={nev}, ncv={ncv}, "
          f"factorization={solver_used}")
    print(f"MUMPS OOC enabled — temp files at: {ooc_dir}")
    E.solve()
    return E


# =====================================================================
# 3. Extract and report results
# =====================================================================
def report_results(E, J, nev, residual_tol=1e-6):
    nconv = E.getConverged()
    nev_requested = E.getDimensions()[0]

    print(f"\nConverged eigenpairs: {nconv} / {nev_requested} requested")
    if nconv < nev_requested:
        print("  WARNING: fewer eigenpairs converged than requested.")
        print("  Consider: increasing ncv, increasing max_it, or checking")
        print("  whether sigma is placed sensibly relative to the spectrum.")

    vr, vi = J.createVecs()
    results = []
    for i in range(nev):
        val = E.getEigenpair(i, vr, vi)
        err = E.computeError(i)
        results.append({
            'eigenvalue': val,
            'residual': err,
            'vec_real': vr.getArray().copy(),
            'vec_imag': vi.getArray().copy(),
        })

    # sort by largest growth rate (real part), descending
    # results.sort(key=lambda r: -r['eigenvalue'].real)
    top_results = results[:nev]

    print(f"\n{'#':>3} {'Re(lambda)':>14} {'Im(lambda)':>14} {'residual':>12}  status")
    print("-" * 66)
    for i, r in enumerate(top_results):
        lam = r['eigenvalue']
        err = r['residual']
        if err > residual_tol:
            status = "UNRELIABLE (residual above tol)"
        elif lam.real > 0:
            status = "UNSTABLE"
        elif abs(lam.real) < 1e-3:
            status = "MARGINAL (near Re=0 -- check carefully)"
        else:
            status = "stable"
        print(f"{i:>3} {lam.real:>14.6f} {lam.imag:>14.6f} {err:>12.2e}  {status}")

    return top_results


# =====================================================================
# Print ALL converged eigenvalues (not just top 10)
# =====================================================================
def print_eigenvalues(results, residual_tol=1e-6):
    print(f"\n{len(results)} eigenvalues:")
    print(f"{'#':>3} {'Re(lambda)':>14} {'Im(lambda)':>14} {'residual':>12}  status")
    print("-" * 66)
    for i, r in enumerate(results):
        lam = r['eigenvalue']
        err = r['residual']
        if err > residual_tol:
            status = "UNRELIABLE (residual above tol)"
        elif lam.real > 0:
            status = "UNSTABLE"
        elif abs(lam.real) < 1e-3:
            status = "MARGINAL (near Re=0)"
        else:
            status = "stable"
        print(f"{i:>3} {lam.real:>14.6f} {lam.imag:>14.6f} {err:>12.2e}  {status}")


# =====================================================================
# Plot ALL converged eigenvalues on the complex plane
# =====================================================================
def plot_eigenspectrum(results, residual_tol=1e-6, save_path='eigenspectrum.png'):
    eigs = np.array([r['eigenvalue'] for r in results])
    residuals = np.array([r['residual'] for r in results])
    reliable = residuals <= residual_tol

    fig, ax = plt.subplots(figsize=(7.5, 6))

    ax.axvspan(min(eigs.real.min(), -0.1) - 0.05, 0, color='tab:blue', alpha=0.06)
    ax.axvspan(0, max(eigs.real.max(), 0.1) + 0.05, color='tab:red', alpha=0.06)
    ax.axvline(0, color='black', lw=1.0)

    ax.scatter(eigs.real[reliable], eigs.imag[reliable],
               c='tab:blue', s=45, edgecolor='white', linewidth=0.5,
               label='converged (residual OK)', zorder=3)
    if (~reliable).any():
        ax.scatter(eigs.real[~reliable], eigs.imag[~reliable],
                   c='gray', s=45, marker='x',
                   label='residual above tolerance -- do not trust', zorder=3)

    # annotate the leading (largest growth rate) eigenvalue
    idx_lead = np.argmax(eigs.real)
    ax.annotate(f'lambda = {eigs[idx_lead].real:.4f} + {eigs[idx_lead].imag:.4f}j',
                xy=(eigs[idx_lead].real, eigs[idx_lead].imag),
                xytext=(10, 10), textcoords='offset points', fontsize=8)

    ax.set_xlabel('Re(lambda)  (growth rate)')
    ax.set_ylabel('Im(lambda)  (frequency, rad/s or non-dim)')
    ax.set_title('Eigenspectrum of the global flux Jacobian')
    ax.legend(fontsize=8, loc='best')
    ax.grid(alpha=0.2)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.show()
    print(f"\nSaved: {save_path}")

# =====================================================================
# Save eigenvalues + eigenvectors in a portable format (.npz)
# =====================================================================
def save_eigendata(E, J, sigma, out_path='eigendata.npz'):
    """
    Saves:
      eigenvalues : complex array, shape (nconv,)
      eigenvectors: complex array, shape (nconv, N) -- each row is one
                    eigenvector, full length N (matches Jacobian dimension)
      residuals   : real array, shape (nconv,)
      sigma       : the complex shift used for this solve
      nev, ncv    : solver settings used, for reproducibility
    """
    nconv = E.getConverged()
    N = J.getSize()[0]
    nev_requested, ncv_used, _ = E.getDimensions()

    vr, vi = J.createVecs()
    eigenvalues = np.zeros(nconv, dtype=complex)
    eigenvectors = np.zeros((nconv, N), dtype=complex)
    residuals = np.zeros(nconv)

    for i in range(nconv):
        val = E.getEigenpair(i, vr, vi)
        eigenvalues[i] = val
        residuals[i] = E.computeError(i)
        # PETSc splits real/imag into separate vecs even in a complex
        # build when using getEigenpair this way -- combine them here
        eigenvectors[i, :] = vr.getArray() + 1j * vi.getArray()

    np.savez(out_path,
             eigenvalues=eigenvalues,
             eigenvectors=eigenvectors,
             residuals=residuals,
             sigma=np.array([sigma]),
             nev=nev_requested,
             ncv=ncv_used)

    print(f"Saved {nconv} eigenpairs to {out_path}")
    print(f"  eigenvalues.shape  = {eigenvalues.shape}")
    print(f"  eigenvectors.shape = {eigenvectors.shape}  (row i = eigenvector for eigenvalues[i])")
    
# =====================================================================
# Interpretation guide (printed, not just code) -- read this after running
# =====================================================================
def print_interpretation_guide():
    print("""
--------------------------------------------------------------------
How to interpret this output:

1. Re(lambda) > 0  -> that mode grows in time -> globally unstable.
   Re(lambda) < 0  -> decays -> stable.
   Re(lambda) ~ 0  -> marginal; this is the Hopf-relevant regime.

2. A genuine Hopf bifurcation shows up as a COMPLEX-CONJUGATE PAIR
   (nonzero Im(lambda), and you should see its conjugate elsewhere
   in the list or in a re-run with wider nev) with Re(lambda) crossing
   zero as you vary your control parameter (Reynolds number). A real
   eigenvalue crossing zero alone would indicate a different
   (steady/pitchfork) bifurcation, not Hopf.

3. Trust ONLY eigenpairs with residual comfortably below your solver
   tolerance (1e-6 to 1e-8 is typical). An eigenvalue with residual
   above tolerance is not converged -- don't draw physical conclusions
   from it, especially near Re(lambda)=0 where you need real precision.

4. If nconv < nev requested: SLEPc could not converge everything you
   asked for within max_it iterations at this ncv. Don't assume the
   unconverged ones don't exist -- widen ncv/max_it and re-run before
   concluding anything about the missing eigenvalues.

5. Sanity checks before trusting a Hopf conclusion:
   - Increase ncv and re-run: do the top eigenvalues change more than
     your tolerance? If yes, ncv was too small.
   - Nudge sigma slightly and re-run: do the same eigenvalues reappear?
     If eigenvalues disappear/appear with small sigma changes, you may
     be missing modes near the edge of what shift-invert "saw".
   - If you have a mesh-refinement study available, confirm the
     leading eigenvalue's real part doesn't move significantly under
     refinement -- a Hopf point that moves with mesh resolution isn't
     trustworthy yet.
--------------------------------------------------------------------
""")

In [6]:
# read the jacobian
Mesh = 765960
Re   = 60
Mach = 0.2
gamma = 1.4
R_gas = 287.0          # confirm units match the solver

# define path
data_dir = "/home/ahf25/git/flux_jacobian/data/flux_jacobian_assembly_v4/v3_mesh"
# data_dir = "./data/flux_jacobian"
JACOBIAN_PATH = f"{data_dir}/jacobian_cylinder_{Mesh}_Re{Re}_M{Mach}_fd.npz"   # <-- set to your actual file

J_csr = read_jacobian(JACOBIAN_PATH)

# convert CSR matrix to petsc compatible form
J = scipy_csr_to_petsc(J_csr)

Loaded Jacobian: shape=(3829800, 3829800), nnz=114925404, density=7.84e-06


In [7]:
import time

# set the shift SIGMA
f = 9.505 # frequency in Hz
SIGMA = 0.0 + f * 2 * np.pi*1j                            # <-- set your shift here

nev = 10
ncv = 300
t0 = time.perf_counter()
E = solve_shift_invert(J, sigma=SIGMA, nev=nev, ncv = ncv)
# save run time 
t = time.perf_counter() - t0

top_results = report_results(E,J, nev) # reporting the top 10 eigenvalues that are close to SIGMA
print(f"Run time: {t:.2f}s")

Solving: sigma=59.72167634474197j, nev=10, ncv=300, factorization=mumps
MUMPS OOC enabled — temp files at: /mnt/data1/ahf25/


KeyboardInterrupt: 

In [ ]:
# save eigenvectors and eigenvalues
out_dir = "./data/ncv_sweep"
eigen_file = f"eigendata_{Mesh}_Re{Re}_M{Mach}_nev{nev}_ncv{ncv}.npz"

save_eigendata(E, J, sigma=SIGMA, out_path=f"{out_dir}/{eigen_file}")

Saved 77 eigenpairs to ./data/ncv_sweep/eigendata_3060_Re60_M0.2_nev10_ncv300.npz
  eigenvalues.shape  = (77,)
  eigenvectors.shape = (77, 15300)  (row i = eigenvector for eigenvalues[i])


A good way to determine the nev, ncv and SIGMA
- conduct nev and ncv sweeps
- take the lowest ncv with the consistent converged eigenvalues

In [ ]:
# ncv sweep

import time
import os

ncv_list   = [150] #[300, 600] #, 1200, 2400]   # extend as needed
out_dir    = "./data/ncv_sweep"
os.makedirs(out_dir, exist_ok=True)

f     = 9.505
SIGMA = 0.0 + f * 2 * np.pi * 1j
nev   = 10

for ncv in ncv_list:
    print(f"\n{'='*60}")
    print(f"  ncv = {ncv}")
    print(f"{'='*60}")

    t0 = time.perf_counter()
    E  = solve_shift_invert(J, sigma=SIGMA, nev=nev, ncv=ncv)
    runtime = time.perf_counter() - t0

    # ── extract all converged eigenpairs ──────────────────────────────
    nconv = E.getConverged()
    N     = J.getSize()[0]
    vr, vi = J.createVecs()

    eigenvalues  = np.zeros(nconv, dtype=complex)
    eigenvectors = np.zeros((nconv, N), dtype=complex)
    residuals    = np.zeros(nconv)

    for i in range(nconv):
        val = E.getEigenpair(i, vr, vi)
        eigenvalues[i]     = val
        residuals[i]       = E.computeError(i)
        eigenvectors[i, :] = vr.getArray() + 1j * vi.getArray()

    # ── save ──────────────────────────────────────────────────────────
    fname = (f"{out_dir}/eigendata"
             f"_{Mesh}_Re{Re}_M{Mach}"
             f"_nev{nev}_ncv{ncv}.npz")

    np.savez(fname,
             eigenvalues  = eigenvalues,
             eigenvectors = eigenvectors,
             residuals    = residuals,
             sigma        = np.array([SIGMA]),
             nev          = nev,
             ncv          = ncv,
             runtime_s    = runtime)

    print(f"  Converged: {nconv}/{nev}  |  runtime: {runtime:.1f}s")
    print(f"  Saved → {fname}")

In [ ]:
# nev sweep
nev_list   = [20,40]   # extend as needed
out_dir    = "./data/nev_sweep"
os.makedirs(out_dir, exist_ok=True)

f     = 9.505
SIGMA = 0.0 + f * 2 * np.pi * 1j
nev   = 10

for nev in nev_list:
    ncv = nev * 30
    print(f"\n{'='*60}")
    print(f"  nev = {nev}")
    print(f"  ncv = {ncv}")
    print(f"{'='*60}")

    t0 = time.perf_counter()
    E  = solve_shift_invert(J, sigma=SIGMA, nev=nev, ncv=ncv)
    runtime = time.perf_counter() - t0

    # ── extract all converged eigenpairs ──────────────────────────────
    nconv = E.getConverged()
    N     = J.getSize()[0]
    vr, vi = J.createVecs()

    eigenvalues  = np.zeros(nconv, dtype=complex)
    eigenvectors = np.zeros((nconv, N), dtype=complex)
    residuals    = np.zeros(nconv)

    for i in range(nconv):
        val = E.getEigenpair(i, vr, vi)
        eigenvalues[i]     = val
        residuals[i]       = E.computeError(i)
        eigenvectors[i, :] = vr.getArray() + 1j * vi.getArray()

    # ── save ──────────────────────────────────────────────────────────
    fname = (f"{out_dir}/eigendata"
             f"_{Mesh}_Re{Re}_M{Mach}"
             f"_nev{nev}_ncv{ncv}.npz")

    np.savez(fname,
             eigenvalues  = eigenvalues,
             eigenvectors = eigenvectors,
             residuals    = residuals,
             sigma        = np.array([SIGMA]),
             nev          = nev,
             ncv          = ncv,
             runtime_s    = runtime)

    print(f"  Converged: {nconv}/{nev}  |  runtime: {runtime:.1f}s")
    print(f"  Saved → {fname}")

In [ ]:
# SIGMA real part sweep 

f        = 9.505
omega    = f * 2 * np.pi                     # imaginary part fixed
sigma_r_list = [-100, -50, -20, -10, -5, 0, 5, 10, 20, 50, 100]   # s⁻¹

nev = 10
ncv = 600
residual_tol = 1e-6

out_dir = f"./data/sigma_sweep/nev{nev}_ncv{ncv}"
os.makedirs(out_dir, exist_ok=True)

# track the mode closest to (0 + omega*j) across runs
target = 0.0 + omega * 1j

print(f"\n{'Re(σ)':>8} {'Re(λ)':>14} {'Im(λ)':>14} "
      f"{'residual':>12} {'runtime(s)':>12}  status")
print("-" * 70)

for sigma_r in sigma_r_list:
    sigma = sigma_r + omega * 1j

    t0      = time.perf_counter()
    E       = solve_shift_invert(J, sigma=sigma, nev=nev, ncv=ncv)
    runtime = time.perf_counter() - t0

    nconv  = E.getConverged()
    N      = J.getSize()[0]
    vr, vi = J.createVecs()

    eigenvalues  = np.zeros(nconv, dtype=complex)
    eigenvectors = np.zeros((nconv, N), dtype=complex)
    residuals    = np.zeros(nconv)

    for i in range(nconv):
        val = E.getEigenpair(i, vr, vi)
        eigenvalues[i]     = val
        residuals[i]       = E.computeError(i)
        eigenvectors[i, :] = vr.getArray() + 1j * vi.getArray()

    # match to target eigenvalue
    idx = np.argmin(np.abs(eigenvalues - target))
    lam = eigenvalues[idx]
    res = residuals[idx]

    status = ('UNRELIABLE' if res > residual_tol
              else 'UNSTABLE'  if lam.real > 0
              else 'stable')

    print(f"{sigma_r:>8.1f} {lam.real:>14.4f} {lam.imag:>14.4f} "
          f"{res:>12.2e} {runtime:>12.1f}  {status}")

    # save
    fname = (f"{out_dir}/eigendata"
             f"_{Mesh}_Re{Re}_M{Mach}"
             f"_nev{nev}_ncv{ncv}"
             f"_sigR{sigma_r:+.0f}.npz")
    np.savez(fname,
             eigenvalues  = eigenvalues,
             eigenvectors = eigenvectors,
             residuals    = residuals,
             sigma        = np.array([sigma]),
             nev          = nev,
             ncv          = ncv,
             runtime_s    = runtime)

In [ ]:
# report the results

import numpy as np
import os, glob

out_dir      = f"./data/sigma_sweep/nev{nev}_ncv{ncv}"
residual_tol = 1e-5
target       = 0.0 + omega * 1j   # same target used during sweep

# load all .npz files in the directory, sorted by Re(sigma)
npz_files = sorted(glob.glob(f"{out_dir}/eigendata_*.npz"),
                   key=lambda p: float(p.split('_sigR')[1].replace('.npz', '')))
print(f"Residual Tolerance: {residual_tol}")
print(f"nev = {nev}, ncv = {ncv}")
print(f"\n{'Re(σ)':>8} {'Im(σ)':>8} {'Re(λ)':>14} {'Im(λ)':>14} "
      f"{'residual':>12} {'runtime(s)':>12}  status")
print("-" * 70)

for fpath in npz_files:
    d = np.load(fpath, allow_pickle=True)

    eigenvalues = d['eigenvalues']
    residuals   = d['residuals']
    sigma       = complex(d['sigma'][0])
    runtime     = float(d['runtime_s'])
    sigma_r     = sigma.real
    sigma_i     = sigma.imag

    # match to target
    idx = np.argmin(np.abs(eigenvalues - target))
    lam = eigenvalues[idx]
    res = residuals[idx]

    status = ('UNRELIABLE' if res > residual_tol
              else 'UNSTABLE'  if lam.real > 0
              else 'stable')

    print(f"{sigma_r:>8.2f} {sigma_i:>8.2f} {lam.real:>14.4f} {lam.imag:>14.4f} "
          f"{res:>12.2e} {runtime:>12.1f}  {status}")

Residual Tolerance: 1e-05
nev = 10, ncv = 600

   Re(σ)    Im(σ)          Re(λ)          Im(λ)     residual   runtime(s)  status
----------------------------------------------------------------------
 -100.00    59.72        12.7936        51.4926     1.37e-06        201.2  UNSTABLE
  -50.00    59.72        12.7936        51.4926     2.33e-06        275.3  UNSTABLE
  -20.00    59.72        12.7936        51.4926     4.83e-06        235.1  UNSTABLE
  -10.00    59.72        12.7936        51.4926     4.26e-06        252.5  UNSTABLE
   -5.00    59.72        12.7936        51.4926     1.43e-06        302.0  UNSTABLE
    0.00    59.72        12.7936        51.4926     5.30e-06        367.9  UNSTABLE
    5.00    59.72        12.7936        51.4926     3.93e-06        286.9  UNSTABLE
   10.00    59.72        12.7936        51.4926     8.69e-06        253.2  UNSTABLE
   20.00    59.72        12.7936        51.4926     2.41e-05        243.8  UNRELIABLE
   50.00    59.72        11.2244        55

In [ ]:
# post-process one case (the latest case)
top_results = report_results(E,J,nev)
print(f"runtime: {runtime:.2f}s")